# 01 · Data Preparation (AI version)

This is the **Claude-Code-assisted** version of the assignment, kept separate from the hand-written `manual/` solution so their performances can be compared.

Key difference from `manual/`: the categorical column `Product_Info_2` is **encoded as an integer index** (for a learned **embedding** in the model), instead of one-hot. Everything else numeric is median-imputed, missing-flagged, and standardized.

Output: `../data/processed_ai.npz` (loaded by `02_train_eval.ipynb`).

## Part 1: Load and split

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv("../data/train.csv")
split = pd.read_csv("../data/split_indices.csv")
test = pd.read_csv("../data/test.csv")

df = train.merge(split, on="Id")
train_df = df[df["split"] == "train"].drop(columns="split").reset_index(drop=True)
val_df   = df[df["split"] == "val"].drop(columns="split").reset_index(drop=True)

print("train_df:", train_df.shape, " val_df:", val_df.shape, " test:", test.shape)

train_df: (47504, 128)  val_df: (11877, 128)  test: (19765, 127)


## Part 2: Brief EDA recap

(The deep EDA lives in `manual/01_data_prep.ipynb`; here we just confirm the key facts the preprocessing depends on.)

In [2]:
miss = train_df.isnull().mean().sort_values(ascending=False)
print("top missing columns (%):")
print((miss[miss > 0].head(13) * 100).round(1))
print("\ntext columns:", list(train_df.select_dtypes(include="object").columns))
print("Product_Info_2 categories:", sorted(train_df["Product_Info_2"].unique()))

top missing columns (%):
Medical_History_10     99.1
Medical_History_32     98.1
Medical_History_24     93.5
Medical_History_15     75.1
Family_Hist_5          70.4
Family_Hist_3          57.6
Family_Hist_2          48.3
Insurance_History_5    42.6
Family_Hist_4          32.3
Employment_Info_6      18.2
Medical_History_1      14.9
Employment_Info_4      11.4
Employment_Info_1       0.0
dtype: float64

text columns: ['Product_Info_2']
Product_Info_2 categories: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'B1', 'B2', 'C1', 'C2', 'C3', 'C4', 'D1', 'D2', 'D3', 'D4', 'E1']


**Recap:** 3 columns are >90% missing (`Medical_History_10/32/24`, dropped); 13 columns have missing values total; the only text column is `Product_Info_2` (19 categories), which we map to integer codes for an embedding.

## Part 3: Preprocess

- Numeric branch: median impute → standardize (fit on train only), plus missing-flags for columns >10% missing.
- Categorical `Product_Info_2`: integer codes from the training categories only; any unseen/missing category → 0. Kept as a **separate array** for the embedding (not one-hot).

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler

drop_cols = ["Medical_History_10", "Medical_History_32", "Medical_History_24"]
cat_col = "Product_Info_2"
num_cols = [c for c in train_df.columns if c not in (["Id", "Response", cat_col] + drop_cols)]

miss_rate = train_df[num_cols].isnull().mean()
flag_cols = list(miss_rate[miss_rate > 0.10].index)

num_pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), num_cols),
        ("flag", MissingIndicator(features="all"), flag_cols),
    ],
    remainder="drop",
)

Xtr_num = num_pre.fit_transform(train_df).astype("float32")
Xva_num = num_pre.transform(val_df).astype("float32")
Xte_num = num_pre.transform(test).astype("float32")
print("numeric feature matrix:", Xtr_num.shape[1], "columns")

numeric feature matrix: 131 columns


In [4]:
# Integer codes for Product_Info_2 (categories learned from train only).
categories = sorted(train_df[cat_col].dropna().unique())
cat_to_idx = {c: i + 1 for i, c in enumerate(categories)}   # reserve 0 for unknown/missing
n_categories = len(categories) + 1

def encode_cat(s):
    return s.map(cat_to_idx).fillna(0).astype("int64").to_numpy()

cat_tr = encode_cat(train_df[cat_col])
cat_va = encode_cat(val_df[cat_col])
cat_te = encode_cat(test[cat_col])
print("n_categories (incl. unknown=0):", n_categories)
print("example codes:", cat_tr[:8])

n_categories (incl. unknown=0): 20
example codes: [17  1 16  8 16 17 19 17]


In [5]:
# Sanity checks
print("same numeric n_cols  train==val:", Xtr_num.shape[1] == Xva_num.shape[1],
      " train==test:", Xtr_num.shape[1] == Xte_num.shape[1])
print("total missing in numeric matrix:", int(np.isnan(Xtr_num).sum()))
print("standardized numeric mean~0/std~1 (first 4 cols):",
      np.round(Xtr_num[:, :4].mean(0), 3), np.round(Xtr_num[:, :4].std(0), 3))

same numeric n_cols  train==val: True  train==test: True
total missing in numeric matrix: 0
standardized numeric mean~0/std~1 (first 4 cols): [ 0. -0.  0. -0.] [1. 1. 1. 1.]


## Part 4: Save

In [6]:
ytr = train_df["Response"].astype("int64").to_numpy()
yva = val_df["Response"].astype("int64").to_numpy()
test_ids = test["Id"].to_numpy()

np.savez_compressed(
    "../data/processed_ai.npz",
    Xtr_num=Xtr_num, Xva_num=Xva_num, Xte_num=Xte_num,
    cat_tr=cat_tr, cat_va=cat_va, cat_te=cat_te,
    ytr=ytr, yva=yva, test_ids=test_ids,
    n_categories=np.int64(n_categories),
    n_num_features=np.int64(Xtr_num.shape[1]),
)
print("saved -> ../data/processed_ai.npz")
print("num features:", Xtr_num.shape[1], "| n_categories:", n_categories)

saved -> ../data/processed_ai.npz
num features: 131 | n_categories: 20
